# Module 19: Distributed Web Crawler Deduplication Google — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/web_crawler_frontier.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import web_crawler_frontier

classes = [n for n, o in inspect.getmembers(web_crawler_frontier, inspect.isclass)
           if o.__module__ == 'web_crawler_frontier']
functions = [n for n, o in inspect.getmembers(web_crawler_frontier, inspect.isfunction)
             if o.__module__ == 'web_crawler_frontier']

print('module   : web_crawler_frontier')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(web_crawler_frontier, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Url canonicalization and normalization

This is the module's own `test_url_canonicalization_and_normalization` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import pytest
from web_crawler_frontier import (
    RobotsTxtParser,
    URLCanonicalizer,
)

raw = "HTTPS://Example.COM:443/products//phones/?utm_source=fb&z=1&a=2#specs"
canonical = URLCanonicalizer.canonicalize(raw)
assert canonical == "https://example.com/products/phones?a=2&z=1"

# Default port removal for HTTP
assert URLCanonicalizer.canonicalize("http://example.com:80/about/") == "http://example.com/about"

# Invalid scheme throws ValueError
with pytest.raises(ValueError):
    URLCanonicalizer.canonicalize("ftp://example.com/files")

print('PASSED: test_url_canonicalization_and_normalization')

## 3. 🔮 Prediction — commit before you run

Predict what happens to a crawler with no URL frontier deduplication when it encounters two pages that link to each other.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_spider_trap_detection`, which tests exactly this property.


In [ ]:
deep_url = "https://example.com/a/b/c/d/e/f/g/h"
assert URLCanonicalizer.is_spider_trap(deep_url, max_depth=5) is True

# Repeated directory loop
loop_url = "https://example.com/category/books/category/books/category/books"
assert URLCanonicalizer.is_spider_trap(loop_url, max_repeats=2) is True

# Valid normal URL
valid_url = "https://example.com/blog/2026/09/architecture-guide"
assert URLCanonicalizer.is_spider_trap(valid_url) is False

print('PASSED: test_spider_trap_detection')

## 4. Measure it: Robots txt parser

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_robots_txt_parser` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

robots_content = """
User-agent: *
Disallow: /private/
Allow: /private/public-preview
Crawl-delay: 1.5
"""
parser = RobotsTxtParser(robots_content)
assert parser.crawl_delay_sec == 1.5
assert parser.is_allowed("/index.html") is True
assert parser.is_allowed("/private/secret") is False
assert parser.is_allowed("/private/public-preview") is True

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_robots_txt_parser')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(web_crawler_frontier) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. A frontier without dedup revisits forever - politeness and dedup are the same concern.
2. Content hashing catches near-duplicates that URL comparison misses.
3. robots.txt and per-host rate limits are correctness requirements, not etiquette.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
